In [1]:
!pip install transformers datasets==2.19.0 accelerate evaluate scipy scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 8.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("masharma/convolearn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/2134 [00:00<?, ? examples/s]

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "allenai/longformer-base-4096"
)

def count_tokens(example):
    return {
        "num_tokens": len(
            tokenizer(
                example["cleaned_conversation"],
                truncation=False
            )["input_ids"]
        )
    }

token_lengths = dataset["train"].map(count_tokens)

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2134 [00:00<?, ? examples/s]

In [5]:
from datasets import DatasetDict
import numpy as np

raw = dataset["train"]

# Group row indices by topic
topic_to_indices = {}
for i, topic in enumerate(raw["earthscience_topic"]):
    if topic not in topic_to_indices:
        topic_to_indices[topic] = []
    topic_to_indices[topic].append(i)

print("Topic distribution:")
for topic, indices in topic_to_indices.items():
    print(f"  {topic}: {len(indices)} rows")

# For each topic, split its rows 70/15/15
train_indices, val_indices, test_indices = [], [], []

rng = np.random.default_rng(seed=42)

for topic, indices in topic_to_indices.items():
    indices = np.array(indices)
    rng.shuffle(indices)

    n = len(indices)
    n_train = int(0.70 * n)
    n_val   = int(0.15 * n)

    train_indices.extend(indices[:n_train].tolist())
    val_indices.extend(indices[n_train : n_train + n_val].tolist())
    test_indices.extend(indices[n_train + n_val :].tolist())

# Build splits
train_ds = raw.select(train_indices)
val_ds   = raw.select(val_indices)
test_ds  = raw.select(test_indices)

dataset_dict = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})


Topic distribution:
  Earth's Energy: 794 rows
  Investigation and Experimentation: 365 rows
  Solid Earth: 371 rows
  Astronomy and Cosmology: 604 rows


In [6]:
def format_conversation(example):
    text = (
        f"Dimension: {example['kb_dim']}\n"
        f"{example['cleaned_conversation']}"
    )
    return {"input_text": text}

dataset_dict = dataset_dict.map(format_conversation)

Map:   0%|          | 0/1491 [00:00<?, ? examples/s]

Map:   0%|          | 0/318 [00:00<?, ? examples/s]

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")

def tokenize(example):
    encoding = tokenizer(
        example["input_text"],
        max_length=4096,
        truncation=True,
        padding=False,  # no padding here — done per batch instead
    )
    encoding["global_attention_mask"] = [0] * len(encoding["input_ids"])
    encoding["global_attention_mask"][0] = 1
    encoding["labels"] = float(example["effectiveness_consensus"])
    return encoding

tokenized = dataset_dict.map(
    tokenize,
    remove_columns=dataset_dict["train"].column_names,
)

print(tokenized)
print("Sample keys:", list(tokenized["train"].features.keys()))
print("Labels (first 3):", tokenized["train"]["labels"][:3])

Map:   0%|          | 0/1491 [00:00<?, ? examples/s]

Map:   0%|          | 0/318 [00:00<?, ? examples/s]

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels'],
        num_rows: 1491
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels'],
        num_rows: 318
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels'],
        num_rows: 325
    })
})
Sample keys: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels']
Labels (first 3): [2.5, 4.5, 4.0]


In [8]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [9]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import torch

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/longformer-base-4096",
    num_labels=1,
    problem_type="regression",
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./longformer-convolearn",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    seed=42,
    gradient_checkpointing=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

[transformers] LongformerForSequenceClassification LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.dense.weight           | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
lm_head.decoder.weight         | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
lm_head.bias                   | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias             | UNEXPECTED | 
classifier.dense.bias          | MISSING    | 
classifier.dense.weight        | MISSING    | 
classifier.out_proj.weight     | MISSING    | 
classifier.out_proj.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

Epoch,Training Loss,Validation Loss
1,7.281095,0.841893
2,5.398025,0.607664
3,4.533194,0.605413
4,3.944788,0.737396
5,2.329539,0.645148


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=935, training_loss=7.92479380235315, metrics={'train_runtime': 2768.9409, 'train_samples_per_second': 2.692, 'train_steps_per_second': 0.338, 'total_flos': 3418532040495870.0, 'train_loss': 7.92479380235315, 'epoch': 5.0})

In [10]:
import numpy as np
from scipy.stats import pearsonr

predictions = trainer.predict(tokenized["test"])
preds = np.squeeze(predictions.predictions)
labels = predictions.label_ids

pearson_r, p_value = pearsonr(preds, labels)
rmse = np.sqrt(np.mean((preds - labels) ** 2))
mae = np.mean(np.abs(preds - labels))

print(f"Pearson r = {pearson_r:.3f} (p = {p_value:.3g})")
print(f"RMSE = {rmse:.3f}")
print(f"MAE = {mae:.3f}")

Pearson r = 0.672 (p = 4.49e-44)
RMSE = 0.815
MAE = 0.607


In [18]:
import os

model_path = "./models/stage1_final_model"
os.makedirs(model_path, exist_ok=True)

# Colab's local disk is wiped when the runtime disconnects —
# copy this directory to Drive/HF Hub if you need it after this session ends.
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model and tokenizer saved to {model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./models/stage1_final_model


In [19]:
import shutil
from google.colab import files

shutil.make_archive("stage1_final_model", "zip", "models/stage1_final_model")
files.download("stage1_final_model.zip")

KeyboardInterrupt: 

In [21]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import shutil

drive_dir = "/content/drive/MyDrive/PhD/AITutorEval/models"
os.makedirs(drive_dir, exist_ok=True)

# Copies the folder directly — no zip step needed, avoids relying on
# stage1_final_model.zip (which was interrupted above and may be incomplete).
shutil.copytree("models/stage1_final_model", f"{drive_dir}/stage1_final_model", dirs_exist_ok=True)
print(f"Copied to Google Drive: {drive_dir}/stage1_final_model")

Mounted at /content/drive
Copied to Google Drive: /content/drive/MyDrive/PhD/AITutorEval/models/stage1_final_model


### Reloading Stage 1 without retraining

In a future session, once `models/stage1_final_model` exists on disk (or you've copied it back from Drive/HF Hub), skip the data-loading/training cells above entirely and just run:

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "./models/stage1_final_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
```

This restores the exact fine-tuned Stage 1 weights, ready for `trainer.predict(...)` or direct inference — no need to re-run `trainer.train()`.

## Stage 2: multi-head model for individual per-dimension scores

Shared Longformer encoder + one linear output per `kb_dim` (6 heads total), trained with masked MSE so each example only updates the head matching its own dimension. No dimension-conditioning text needed — raw `cleaned_conversation` in, all 6 dimension scores out in a single forward pass. This is a separate model/training run from Stage 1 above; it reuses `dataset_dict`, `tokenizer`, and `raw` but does not depend on Stage 1's trained weights.

In [22]:
ALL_DIMS = sorted(raw.unique("kb_dim"))
DIM2IDX = {dim: i for i, dim in enumerate(ALL_DIMS)}
TARGET_DIMS = ["Cognitive Engagement", "Formative Assessment", "Metacognition"]

print("Dimension -> head index:", DIM2IDX)

def tokenize_multihead(example):
    encoding = tokenizer(
        example["cleaned_conversation"],
        max_length=4096,
        truncation=True,
        padding=False,
    )
    encoding["global_attention_mask"] = [0] * len(encoding["input_ids"])
    encoding["global_attention_mask"][0] = 1
    encoding["labels"] = float(example["effectiveness_consensus"])
    encoding["dim_idx"] = DIM2IDX[example["kb_dim"]]
    return encoding

tokenized_mh = dataset_dict.map(
    tokenize_multihead,
    remove_columns=dataset_dict["train"].column_names,
)

print(tokenized_mh)
print("Sample dim_idx (first 5):", tokenized_mh["train"]["dim_idx"][:5])

Dimension -> head index: {'Accountability': 0, 'Cognitive Engagement': 1, 'Cultural Responsiveness': 2, 'Formative Assessment': 3, 'Metacognition': 4, 'Power Dynamics': 5}


Map:   0%|          | 0/1491 [00:00<?, ? examples/s]

Map:   0%|          | 0/318 [00:00<?, ? examples/s]

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels', 'dim_idx'],
        num_rows: 1491
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels', 'dim_idx'],
        num_rows: 318
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels', 'dim_idx'],
        num_rows: 325
    })
})
Sample dim_idx (first 5): [4, 4, 5, 3, 1]


In [24]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import LongformerModel

class MultiHeadLongformerRegressor(nn.Module):
    def __init__(self, base_model_name, num_dims):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(base_model_name)
        self.heads = nn.Linear(self.longformer.config.hidden_size, num_dims)

    def gradient_checkpointing_enable(self, **kwargs):
        self.longformer.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.longformer.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, global_attention_mask, dim_idx=None, labels=None):
        outputs = self.longformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask,
        )
        pooled = outputs.last_hidden_state[:, 0]  # CLS token representation
        logits = self.heads(pooled)  # (batch, num_dims) — one score per dimension

        loss = None
        if labels is not None and dim_idx is not None:
            selected = logits.gather(1, dim_idx.unsqueeze(1)).squeeze(1)
            loss = F.mse_loss(selected, labels)

        return {"loss": loss, "logits": logits}

mh_model = MultiHeadLongformerRegressor("allenai/longformer-base-4096", num_dims=len(ALL_DIMS))

Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
import inspect
from scipy.stats import pearsonr as _pearsonr

def compute_metrics_multihead(eval_pred):
    preds = eval_pred.predictions  # (N, num_dims)
    labels = np.array(eval_pred.label_ids)  # (N,)
    dim_idx = np.array(eval_pred.inputs["dim_idx"])  # (N,)
    selected = preds[np.arange(len(labels)), dim_idx]

    metrics = {
        "overall_mse": float(np.mean((selected - labels) ** 2)),
        "overall_pearson": float(_pearsonr(selected, labels)[0]),
    }
    for i, name in enumerate(ALL_DIMS):
        mask = dim_idx == i
        if mask.sum() > 1:
            slug = name.lower().replace(" ", "_")
            metrics[f"{slug}_mse"] = float(np.mean((selected[mask] - labels[mask]) ** 2))
            metrics[f"{slug}_pearson"] = float(_pearsonr(selected[mask], labels[mask])[0])
    return metrics

# Newer `transformers` renamed `include_inputs_for_metrics` (bool) to
# `include_for_metrics` (list, e.g. ["inputs"]) — pick whichever this install supports.
ta_params = inspect.signature(TrainingArguments.__init__).parameters
if "include_for_metrics" in ta_params:
    inputs_for_metrics_kwarg = {"include_for_metrics": ["inputs"]}
else:
    inputs_for_metrics_kwarg = {"include_inputs_for_metrics": True}

# Same T4-safe settings as Stage 1: batch size 1, fp16, gradient_checkpointing.
mh_training_args = TrainingArguments(
    output_dir="./longformer-convolearn-multihead",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_overall_mse",
    greater_is_better=False,
    logging_steps=50,
    seed=42,
    gradient_checkpointing=True,
    label_names=["labels"],
    **inputs_for_metrics_kwarg,
)

mh_trainer = Trainer(
    model=mh_model,
    args=mh_training_args,
    train_dataset=tokenized_mh["train"],
    eval_dataset=tokenized_mh["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_multihead,
)

mh_trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
mh_test_metrics = mh_trainer.evaluate(tokenized_mh["test"], metric_key_prefix="test")
print(mh_test_metrics)

In [ ]:
mh_model_path = "./models/stage2_final_model"
os.makedirs(mh_model_path, exist_ok=True)

# Custom nn.Module, not a Trainer-native model class — save state_dict + config manually.
torch.save(mh_model.state_dict(), f"{mh_model_path}/state_dict.pt")
tokenizer.save_pretrained(mh_model_path)

print(f"Stage 2 weights saved to {mh_model_path}/state_dict.pt")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import shutil

drive_dir = "/content/drive/MyDrive/PhD/AITutorEval/models"
os.makedirs(drive_dir, exist_ok=True)

shutil.copytree(mh_model_path, f"{drive_dir}/stage2_final_model", dirs_exist_ok=True)
print(f"Copied to Google Drive: {drive_dir}/stage2_final_model")